In [1]:
# import script and load dataset
import pandas as pd
import numpy as np
from scipy import stats
from flask import render_template,Flask,request
import altair as alt
%matplotlib inline 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error
df=pd.read_csv("GlobalWeatherRepository.csv")


In [2]:
# Clean the dataset and decide the features
#remove null columns
df=df.dropna()
#decide possible relevant columns
cols=["temperature_celsius","country","last_updated","wind_kph","pressure_mb","humidity"]
df=df[cols]
#breakdown date into date only and remove time
df["last_updated"]=pd.to_datetime(df["last_updated"])
df["last_updated_day"]=df["last_updated"].dt.day
df["last_updated_month"]=df["last_updated"].dt.month

#convert country to lower case
nVals=[]
for country in df["country"]:
    nVals.append(country.lower())
df["country"]=nVals
#group dataset by country
Countries=set(df["country"].tolist())
dataDic={}
for row in Countries:
    dataDic[row]={"last_updated":[],"temperature":[],"last_updated_day":[],"last_updated_month":[],"wind_kph":[],"pressure_mb":[],"humidity":[]}
for row in df.itertuples():
    dataDic[row.country]["temperature"].append(row.temperature_celsius)
    dataDic[row.country]["last_updated"].append(row.last_updated)
    dataDic[row.country]["last_updated_day"].append(row.last_updated_day)
    dataDic[row.country]["last_updated_month"].append(row.last_updated_month)
    dataDic[row.country]["wind_kph"].append(row.wind_kph)
    dataDic[row.country]["pressure_mb"].append(row.pressure_mb)
    dataDic[row.country]["humidity"].append(row.humidity)


#deal with outliers
#decided to use the zScore method for this
cols=["temperature","wind_kph","pressure_mb","humidity"]
test=["japan"] #test variable to see if this is working
zVal=2
arDf={}




for country in Countries:
        outDf=pd.DataFrame(dataDic[country])
        indexes=np.where(np.abs(stats.zscore(outDf[cols]))>zVal)
        if(len(indexes[0])>1 and indexes[0].any()!=None):
            outDf=outDf.drop(index=indexes[0])
        dataDic[country]=outDf
        #Modify dataset for timeSeriesPrediction   
for country in Countries:
        arDf[country]=dataDic[country][["temperature"]]


     

#observing correlation to see if any of these columns outside date can be used for analysis    
#research more on autocorrelation and implement it here using the lag values n where n is an element in [1,2,3,7,30] 
for country in test:
    testDf= dataDic[country]
    corr=testDf.corr()
    dataDic[country]=testDf.drop(columns=["wind_kph","last_updated_day"])

#observed low correlation in wind henve dropping it as a potential feature
eDf=dataDic[test[0]]
columns=["temperature","humidity","last_updated_month"]

#fix the skew in mydataset
for country in Countries:
   eDf=dataDic[country]
   for col in columns:
     skew=abs(eDf[col].skew()) 
     if(skew>0):
        eDf[col]=np.log1p(eDf[col])


eDf=dataDic[test[0]]
categories=eDf[(a for a in columns)].drop(columns=["temperature"])
categories=eDf.drop(columns=["temperature","last_updated"])
pairplot=alt.Chart(eDf).mark_circle().encode(
    alt.X(alt.repeat("column"), type='quantitative'),
    alt.Y(alt.repeat("row"), type='quantitative'),
    color='Origin:N'
).properties(
    width=300,
    height=150
).repeat(
    row=['temperature'],
    column=list(categories.columns)
).interactive()

#pressure seems to be the only variable with a usable corelation soi drop over potential columns
#explore other visualization tools

fnDf=eDf[["temperature","humidity"]]
columns=fnDf.columns
          

#N/B:might be ovverfitted as the pressure and humidity per country seemas to vary in correlation
pairplot


c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


alt.RepeatChart(...)

In [3]:
# EDA
#for testing purposes i am using one country first

scaler=MinMaxScaler()
bxDf=pd.DataFrame(scaler.fit_transform(fnDf),columns=columns)
long=bxDf.melt(var_name="feature",value_name="value")
afSkewPlot=alt.Chart(long).mark_boxplot(extent="min-max").encode(
    alt.Y("feature:N").scale(zero=False),
    alt.X("value:Q").scale(zero=False),
    alt.Color("Origin:N").legend(None),
).properties(
    width=400,
    height=300,
    title="Boxplots for data value distribution"
).resolve_scale(  
    y='shared' 
)


linePlot=alt.Chart(bxDf).mark_line().encode(
    x='temperature',
    y='humidity'
).properties(
    width=600,
    title="mapping of humidity to temperature"
).interactive()

fPlots=afSkewPlot&linePlot
fPlots


alt.VConcatChart(...)

In [4]:
# Train Models
#linear Regression
x=pd.DataFrame(fnDf["humidity"])
y=pd.DataFrame(fnDf["temperature"])
x_train,x_test,y_train,y_test=train_test_split(x,y, test_size=0.5,random_state=50)
linReg=LinearRegression().fit(x_train,y_train)
prediction=linReg.predict(x_test)


MAE=mean_absolute_error(y_test,prediction)
MSE=mean_squared_error(y_test,prediction)
RMSE=root_mean_squared_error(y_test,prediction)
print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")





MSE=0.15498713774836584 : MAE=0.31573342090824236 : RMSE=0.3936840582857856


In [5]:
#Auto correlation
#create lag values
from sklearn.preprocessing import PowerTransformer
lagVals=[-1,-2,-3,-7,-30]
for country in Countries:
    for val in lagVals:
        arDf[country][f"temp_lag{abs(val)}"]= arDf[country]["temperature"].shift(val)        
testCountry=test[0]
print(arDf[country].head(5))
transformers={}
for country in Countries:
   arDf[country].dropna(axis=0,how="any",inplace=True,subset=arDf[country].columns)
   eDf=arDf[country]
   columns=arDf[country].columns
   for col in columns:
   
     skew=abs(eDf[col].skew()) 
     if(skew>0):  
       pt = PowerTransformer(method='yeo-johnson')
       if(col=="temperature"):
         transformers[country]=pt
       eDf[col]= pt.fit_transform(eDf[[col]])
       
   arDf[country]=eDf

tArDf=arDf[testCountry]
tArDf.dropna(axis=0,how="any",inplace=True,subset=tArDf.columns)







scaler=MinMaxScaler()
bxDf=pd.DataFrame(tArDf,columns=columns)
long=bxDf.melt(var_name="feature",value_name="value")

afSkewPlot=alt.Chart(long).mark_boxplot(extent="min-max").encode(
    alt.Y("feature:N").scale(zero=False),
    alt.X("value:Q").scale(zero=False),
    alt.Color("Origin:N").legend(None),
).properties(
    width=600,
    height=300,
    title="Boxplots for data value distribution"
).resolve_scale(  
    y='shared' 
)

corr_melted = bxDf.corr().reset_index().melt(id_vars='index')
corr_melted.columns = ['var1', 'var2', 'correlation']
plt.figure(figsize=(15,6))
corrPlot=alt.Chart(corr_melted).mark_rect().encode(
    x='var1:N',
    y='var2:N',
    color=alt.Color('correlation:Q', 
                    scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title="Pearson Corr")),
    tooltip=['var1', 'var2', 'correlation']
).properties(
    width=600,
    height=400,
    title="Feature Correlation Heatmap"
)
text = corrPlot.mark_text().encode(
    text=alt.Text('correlation:Q', format='.2f'),
    color=alt.condition(
        "datum.correlation > 0.5 || datum.correlation < -0.5",
        alt.value('white'),
        alt.value('black')
    )
)

# Combine and show
heatCor=(corrPlot+ text)

plots=heatCor&afSkewPlot
plots


   temperature  temp_lag1  temp_lag2  temp_lag3  temp_lag7  temp_lag30
0         22.5       16.4       15.0       15.5       16.6        24.3
2         16.4       15.0       15.5       12.3       15.4        29.6
3         15.0       15.5       12.3       14.5       19.3        27.2
4         15.5       12.3       14.5       13.0        8.8        24.4
5         12.3       14.5       13.0       16.6       15.4        27.3


alt.VConcatChart(...)

<Figure size 1500x600 with 0 Axes>

In [6]:
#Linear Regression
aY=pd.DataFrame(tArDf["temperature"])
aX=pd.DataFrame(tArDf.drop(columns=["temperature"]))
x_train,x_test,y_train,y_test=train_test_split(aX,aY, test_size=0.2,shuffle=False)
linReg=LinearRegression().fit(x_train,y_train)
prediction=linReg.predict(x_test)

MAE=mean_absolute_error(y_test,prediction)
MSE=mean_squared_error(y_test,prediction)
RMSE=root_mean_squared_error(y_test,prediction)
print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")





MSE=0.16000710212772287 : MAE=0.329171420806258 : RMSE=0.40000887756113973


In [7]:
aY=pd.DataFrame(tArDf["temperature"])
aX=pd.DataFrame(tArDf.drop(columns=["temperature"]))
x_train,x_test,y_train,y_test=train_test_split(aX,aY, test_size=0.2,shuffle=False)
forReg=RandomForestRegressor().fit(x_train,y_train)
prediction=forReg.predict(x_test)
MAE=mean_absolute_error(y_test,prediction)
MSE=mean_squared_error(y_test,prediction)
RMSE=root_mean_squared_error(y_test,prediction)
print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")


MSE=0.18140351385377376 : MAE=0.3367119757950495 : RMSE=0.4259149138663423


c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [8]:
aY=pd.DataFrame(tArDf["temperature"])
aX=pd.DataFrame(tArDf.drop(columns=["temperature"]))
x_train,x_test,y_train,y_test=train_test_split(aX,aY, test_size=0.2,shuffle=False)
xReg=XGBRegressor().fit(x_train,y_train)
prediction=xReg.predict(x_test)

MAE=mean_absolute_error(y_test,prediction)
MSE=mean_squared_error(y_test,prediction)
RMSE=root_mean_squared_error(y_test,prediction)
print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")


MSE=0.2143179476261139 : MAE=0.370532363653183 : RMSE=0.4629448652267456


In [9]:
# look through PACR
# dont use common visualization tools
# create flask app
# hyper parameter tuning
# kfold cross validation




In [10]:
# flask app need to create 2 functions 
# one for training once counrty is inputted and providing stats
# one for prdiciting once the models are trained


In [11]:
# made methods to use in the flask app next need to change visualization tool
from datetime import date,timedelta
setCountry=""
def getStats(country):
    global setCountry
    global  arDf
    setCountry=country
    statsDf=arDf[country]
    long=statsDf.melt(var_name="feature",value_name="value")
    afSkewPlot=alt.Chart(long).mark_boxplot(extent="min-max").encode(
    alt.Y("feature:N").scale(zero=False),
    alt.X("value:Q").scale(zero=False),
    alt.Color("Origin:N").legend(None),
    ).properties(
    width=600,
    height=300,
    title="Boxplots for data value distribution"
    ).resolve_scale(  
        y='shared' )

    corr_melted = statsDf.corr().reset_index().melt(id_vars='index')
    corr_melted.columns = ['var1', 'var2', 'correlation']

    corrPlot=alt.Chart(corr_melted).mark_rect().encode(
    x='var1:N',
    y='var2:N',
    color=alt.Color('correlation:Q', 
                    scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title="Pearson Corr")),
    tooltip=['var1', 'var2', 'correlation']
     ).properties(
    width=600,
    height=400,
    title="Feature Correlation Heatmap"
     )
    text = corrPlot.mark_text().encode(
    text=alt.Text('correlation:Q', format='.2f'),
    color=alt.condition(
        "datum.correlation > 0.5 || datum.correlation < -0.5",
        alt.value('white'),
        alt.value('black')
    )
)

# Combine and show
    heatCor=(corrPlot+ text)

    plots=heatCor&afSkewPlot
    return plots   
def train(country):
    vals={"LinearRegression":[],"RandomForest":[],"XGboost":[]}
    global tArDf
    global aY
    global aX
    tArDf=arDf[country]
    tArDf.dropna(axis=0,how="any",inplace=True,subset=tArDf.columns)
    columns=tArDf.columns  
    aY=pd.DataFrame(tArDf["temperature"])
    aX=pd.DataFrame(tArDf.drop(columns=["temperature"]))
    x_train,x_test,y_train,y_test=train_test_split(aX,aY, test_size=0.2,shuffle=False)
    global linReg
    linReg=LinearRegression().fit(x_train,y_train) 
    prediction=linReg.predict(x_test)
    MAE=mean_absolute_error(y_test,prediction)
    MSE=mean_squared_error(y_test,prediction)
    RMSE=root_mean_squared_error(y_test,prediction)
    print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")
    vals["LinearRegression"]=[MAE,MSE,RMSE]
    global forReg
    forReg=RandomForestRegressor().fit(x_train,y_train)
    prediction=forReg.predict(x_test)
    MAE=mean_absolute_error(y_test,prediction)
    MSE=mean_squared_error(y_test,prediction)
    RMSE=root_mean_squared_error(y_test,prediction)
    vals["RandomForest"]=[MAE,MSE,RMSE]
    print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")
    x_train,x_test,y_train,y_test=train_test_split(aX,aY, test_size=0.2,shuffle=False)
    xReg=XGBRegressor().fit(x_train,y_train)
    prediction=xReg.predict(x_test)
    MAE=mean_absolute_error(y_test,prediction)
    MSE=mean_squared_error(y_test,prediction)
    RMSE=root_mean_squared_error(y_test,prediction)
    print(f"MSE={MSE} : MAE={MAE} : RMSE={RMSE}")
    vals["XGboost"]=[MAE,MSE,RMSE]
    return vals

first_day=date(2024,5,16)
def updateData(index):
    global aY
    aXList=aY["temperature"].to_list()

    for i in range(index):
        feature=np.array([aXList[-1],aXList[-2],aXList[-3],aXList[-7],aXList[-30]])
        feature=feature.reshape((1, -1))
        pred=linReg.predict(feature)
        y_scalar =pred.item() 
        aXList.append(y_scalar)
    
    aY = pd.DataFrame({"temperature": aXList})
    print(aY.head(5))

     
#finish this method correctly
def predict(toDate,algorithim,endDate):
    temps=[]
    global aY
    global transformers
    toDate=date.fromisoformat(toDate)
    endDate=date.fromisoformat(endDate)#might not need this depending on form
    ans=None
    index=(toDate-first_day).days
    days=(endDate-toDate).days
    aXList=aY["temperature"].to_list()
    previous=len(aXList)-index
    if(previous<-2):
        updateData(abs(previous))
    if(previous==-1):
        print("ended here")
        return aXList[index]
    
    aXList=aY["temperature"].to_list() 
    for i in range(days):
        tDay=(toDate+timedelta(i))  
        feature=np.array([aXList[index-1],aXList[index-2],aXList[index-3],aXList[index-7],aXList[index-30]])
        print(feature)
        feature=feature.reshape((1, -1))
        print(setCountry)
    
        match algorithim:
            case "LinearRegression":
                rawans=linReg.predict(feature)
                ans=transformers[setCountry].inverse_transform(rawans.reshape(-1,1))
                temps.append(f"{tDay} : {round(ans.item(),2)}")
            case "RandomForest":
                rawans=forReg.predict(feature)
                ans=transformers[setCountry].inverse_transform(rawans.reshape(-1,1))
                temps.append(f"{tDay} : {round(ans.item(),2)}")
            case "XGboost":
                rawans=xReg.predict(feature)
                ans=transformers[setCountry].inverse_transform(rawans.reshape(-1,1))
                temps.append(f"{tDay} : {round(ans.item(),2)}")
        aXList.append(rawans.item())
        index+=1

    aY = pd.DataFrame({"temperature": aXList})
    return temps

        

    


In [ ]:
#creating the app using flask 

errors={}
app=Flask(__name__)
model=''
@app.route('/')
def firstPage():
    columns={"Title":"Welcome to my tempPredictor",
             "action":"/",
             "inputs":[{"name": "country", "label": "country", "type": "text"}] ,
             "select":{"name":"model","options":["LinearRegression","RandomForest","XGboost"]}}
    return render_template("form.html",data=columns)
@app.post('/')
def statsPage():
    Tcountry=request.form.get("country").lower()
    global model
    model=request.form.get("model")
    myPlots=getStats(Tcountry)
    saved=myPlots.to_json()
    plots={"path":saved}
    global errors
    errors=train(setCountry)
    return render_template("display.html",data=plots)

@app.route('/predict')
def predictPage():
    columns={"Title":f"Enter date and model to predict Temp for{setCountry}  ",
             "action":"/predict",
             "inputs":[ {"name": "date", "label": "date", "type": "date"}, {"name": "end-date", "label": "end-date", "type": "date"}]}
    return render_template("form.html",data=columns)
@app.post('/predict')
def predictresp():
    date=request.form.get("date")
    endDate=request.form.get("end-date")
    global model
    ans=predict(date,model,endDate)
    plots={"temp":ans}
    return render_template("response.html",data=plots)


if __name__ == "__main__":
    try:
        app.run(port=5001, debug=True, use_reloader=False)
    except SystemExit:
        print("Flask server stopped.")
    


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [10/Apr/2026 10:07:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Apr/2026 10:07:18] "GET /favicon.ico HTTP/1.1" 404 -


MSE=0.16000710212772287 : MAE=0.329171420806258 : RMSE=0.40000887756113973


c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
127.0.0.1 - - [10/Apr/2026 10:07:25] "POST / HTTP/1.1" 200 -


MSE=0.1734013126313776 : MAE=0.3291778895057691 : RMSE=0.4164148323863808
MSE=0.2143179476261139 : MAE=0.370532363653183 : RMSE=0.4629448652267456


...

MSE=0.30586565295175233 : MAE=0.4445556130388228 : RMSE=0.5530512209115466
MSE=0.4002540039191342 : MAE=0.514639881306899 : RMSE=0.6326563078948428


c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
127.0.0.1 - - [10/Apr/2026 10:07:52] "POST / HTTP/1.1" 200 -


MSE=0.45791348814964294 : MAE=0.556301474571228 : RMSE=0.6766930818557739


127.0.0.1 - - [10/Apr/2026 10:07:54] "GET /predict HTTP/1.1" 200 -
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\samue\OneDrive\Documents\WBLprojs\.venv\Lib\site

   temperature
0     0.429249
1     1.214619
2     0.027695
3     1.599057
4    -0.380241
[0.21917953 0.21874931 0.2183107  0.2164691  0.20261976]
kenya
[0.21960153 0.21917953 0.21874931 0.21694291 0.2033582 ]
kenya
[0.22001546 0.21960153 0.21917953 0.21740768 0.20408256]
kenya
[0.22042149 0.22001546 0.21960153 0.21786355 0.20479303]
kenya
[0.22081975 0.22042149 0.22001546 0.2183107  0.20548991]
kenya
[0.2212104  0.22081975 0.22042149 0.21874931 0.20617345]
kenya
[0.22159358 0.2212104  0.22081975 0.21917953 0.20684394]
kenya
[0.22196944 0.22159358 0.2212104  0.21960153 0.20750163]
kenya
[0.22233811 0.22196944 0.22159358 0.22001546 0.20814679]
kenya
[0.22269974 0.22233811 0.22196944 0.22042149 0.20877965]
kenya
[0.22305446 0.22269974 0.22233811 0.22081975 0.2094004 ]
kenya
[0.2234024  0.22305446 0.22269974 0.2212104  0.2100093 ]
kenya
[0.22374369 0.2234024  0.22305446 0.22159358 0.21060654]
kenya
[0.22407846 0.22374369 0.2234024  0.22196944 0.21119236]
kenya
